In [6]:
from lab8_full_utils import (
    get_latest_vintage_name,
    get_latest_data,
    load_append,
    load_trunc,
    load_incremental,
    show_table,
)

In [10]:
from pathlib import Path

BASE_DIR = Path.cwd()
SOURCE_FILE = BASE_DIR / "data" / "pcpi_vintages.csv"
print(SOURCE_FILE)

/Users/yirange/computing/yiran-emily-api/lab_08/data/pcpi_vintages.csv


In [12]:
import pandas as pd

df = pd.read_csv(SOURCE_FILE)
print(df.head())
print(df.columns[:10])

      DATE  PCPI98M11  PCPI98M12  PCPI99M1  PCPI99M2  PCPI99M3  PCPI99M4  \
0  1947:01        NaN        NaN       NaN       NaN       NaN       NaN   
1  1947:02        NaN        NaN       NaN       NaN       NaN       NaN   
2  1947:03        NaN        NaN       NaN       NaN       NaN       NaN   
3  1947:04        NaN        NaN       NaN       NaN       NaN       NaN   
4  1947:05        NaN        NaN       NaN       NaN       NaN       NaN   

   PCPI99M5  PCPI99M6  PCPI99M7  ...  PCPI25M5  PCPI25M6  PCPI25M7  PCPI25M8  \
0       NaN       NaN       NaN  ...      21.5      21.5      21.5      21.5   
1       NaN       NaN       NaN  ...      21.6      21.6      21.6      21.6   
2       NaN       NaN       NaN  ...      22.0      22.0      22.0      22.0   
3       NaN       NaN       NaN  ...      22.0      22.0      22.0      22.0   
4       NaN       NaN       NaN  ...      22.0      22.0      22.0      22.0   

   PCPI25M9  PCPI25M10  PCPI25M11  PCPI25M12  PCPI26M1  PCPI26

In [14]:
print(get_latest_vintage_name("2004-01-15"))
print(get_latest_vintage_name("2004-02-15"))
print(get_latest_vintage_name("2004-03-15"))

PCPI04M1
PCPI04M2
PCPI04M3


In [15]:
get_latest_data("2004-03-15").head()

,date,cpi
0,1947:01,21.5
1,1947:02,21.6
2,1947:03,22.0
3,1947:04,22.0
4,1947:05,22.0


In [16]:
load_append("2004-01-15", init=True)
load_trunc("2004-01-15")
load_incremental("2004-01-15", init=True)

load_append("2004-02-15")
load_trunc("2004-02-15")
load_incremental("2004-02-15")

load_append("2004-03-15")
load_trunc("2004-03-15")
load_incremental("2004-03-15")

In [17]:
show_table("cpi_append").head()
show_table("cpi_trunc").head()
show_table("cpi_inc").head()

,date,cpi
0,1947:01,21.5
1,1947:02,21.6
2,1947:03,22.0
3,1947:04,22.0
4,1947:05,22.0


In [19]:
append_times = []
trunc_times = []
inc_times = []

load_append("2024-01-01", init=True)
load_trunc("2024-01-01")
load_incremental("2024-01-01", init=True)

for d in pull_dates[1:]:
    date_str = d.strftime("%Y-%m-%d")

    start = time.perf_counter()
    load_append(date_str)
    append_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    load_trunc(date_str)
    trunc_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    load_incremental(date_str)
    inc_times.append(time.perf_counter() - start)

In [20]:
print("append total:", sum(append_times))
print("trunc total:", sum(trunc_times))
print("inc total:", sum(inc_times))

print("append avg:", sum(append_times) / len(append_times))
print("trunc avg:", sum(trunc_times) / len(trunc_times))
print("inc avg:", sum(inc_times) / len(inc_times))

append total: 10.201738636067603
trunc total: 9.613841391343158
inc total: 8.81263476566528
append avg: 0.02406070433034812
trunc avg: 0.022674154224865937
inc avg: 0.020784515956757735


In [21]:
expected_df = get_latest_data("2025-02-28").reset_index(drop=True)

trunc_df = show_table("cpi_trunc").reset_index(drop=True)
inc_df = show_table("cpi_inc").reset_index(drop=True)

print("trunc correct:", trunc_df.equals(expected_df))
print("inc correct:", inc_df.equals(expected_df))

trunc correct: True
inc correct: True


In [22]:
import duckdb

conn = duckdb.connect("lab8_full.duckdb")

append_last = conn.execute("""
    SELECT date, cpi
    FROM cpi_append
    WHERE pull_date = DATE '2025-02-28'
    ORDER BY date
""").df().reset_index(drop=True)

conn.close()

print("append last snapshot correct:", append_last.equals(expected_df))
print("append last snapshot rows:", len(append_last))
print("expected rows:", len(expected_df))

append last snapshot correct: True
append last snapshot rows: 937
expected rows: 937


## Discussion

After running the simulation, I found that all three methods can work, but they behave differently depending on what we want the database to represent.

For `trunc`, the result was always easy to understand. Each time it runs, it deletes the old table and reloads the newest available data. Because of that, the final table matched the expected data exactly. This method feels the most straightforward, especially for someone who is still learning, because it is simple to check and simple to debug. The downside is that it reloads everything every time, even if only a small part of the data changed.

For `incremental`, the final result was also correct. It matched the expected data just like `trunc`, but it was slightly faster in my simulation. This makes sense because it only updates changed rows and inserts new ones, instead of rewriting the whole table. At first, this method felt harder to understand because the logic is more detailed, but after seeing the result, it seems like a more practical method if the dataset keeps growing over time.

For `append`, I noticed that it depends on how we think about the table. If I look at the whole append table, it keeps growing because it stores a snapshot for each pull date. In that sense, it is useful for keeping history. But if I only look at the last snapshot, it also matched the expected data. So this method is not really “wrong,” but it serves a different purpose. It is better for preserving historical pulls, while `trunc` and `incremental` are better if the goal is to maintain one current version of the database.

From the runtime results, `incremental` was the fastest on average, `trunc` was next, and `append` was the slowest in this simulation. The differences were not huge, but they were still noticeable. This helped me see that there is usually a tradeoff between simplicity, storage, and efficiency.

Overall, this lab helped me understand that data loading is not only about getting data into a database. It is also about deciding what kind of database you want to maintain. If I only care about the latest correct version, `trunc` and `incremental` make more sense. If I want to keep a record of what the data looked like on each pull date, `append` is useful. Personally, after doing this lab, I feel that `incremental` is the most balanced method because it keeps the final table correct without reloading everything every time.